# Opportunity Agent

Looks through the SAP article chunks and pulls out business opportunities
(new markets, partnerships, AI/tech opportunities, product opportunities).

Uses the same FAISS index built in `embedding_v2.ipynb`, and a local Ollama
model to do the reasoning (no Anthropic/OpenAI allowed for this project).


In [1]:
import json
import os
import time

import faiss
import pandas as pd
import requests
from sentence_transformers import SentenceTransformer
from pathlib import Path

# find the data folder whether we run from agents/ or repo root
cwd = Path.cwd()
DATA_DIR = next((p for p in [cwd / "notebook" / "data", cwd.parent / "notebook" / "data"] if p.exists()), cwd)

CHUNKS_PATH = DATA_DIR / "chunked_data.json"
INDEX_PATH  = DATA_DIR / "sap_intelligence.index"
OUT_PATH    = DATA_DIR / "opportunities.json"

EMBED_MODEL  = "BAAI/bge-small-en-v1.5"
OLLAMA_HOST  = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "phi4-mini")
TIMEOUT      = int(os.getenv("OLLAMA_TIMEOUT", "900"))  # local CPU models are slow, give it time

print("data dir:", DATA_DIR)


data dir: /workspaces/AI-CEO-Strategic-Intelligence-Agent/notebook/data


In [2]:
# load everything we built in embedding_v2.ipynb
chunks_df = pd.read_json(CHUNKS_PATH)
index = faiss.read_index(str(INDEX_PATH))
model = SentenceTransformer(EMBED_MODEL)

print("chunks loaded:", len(chunks_df))
print("vectors in index:", index.ntotal)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

chunks loaded: 1387
vectors in index: 1387


In [ ]:
def retrieve(query, k=8):
    q_vec = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q_vec, k)
    results = chunks_df.iloc[idx[0]]
    return results["text"].tolist()


def ask_llm(prompt):
    resp = requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
        timeout=TIMEOUT,
    )
    resp.raise_for_status()
    return resp.json()["response"]


def warmup():
    print("warming up model...")
    start = time.time()
    ask_llm("hi")
    print("warmup done in", round(time.time() - start, 1)/60, "sec")


warmup()


warming up model...
warmup done in 263.8 sec


## Build context

Pull the most relevant chunks for a few opportunity-related questions and
combine them into one block of text. This becomes the "evidence" the LLM
reasons over.


In [5]:
queries = [
    "SAP new markets and partnerships",
    "SAP AI products and innovation",
    "SAP revenue growth opportunities",
]

chunks = []
for q in queries:
    chunks.extend(retrieve(q, k=6))

# remove exact duplicate chunks (same chunk can show up for multiple queries)
chunks = list(dict.fromkeys(chunks))

context = "\n\n".join(chunks)
print("number of chunks used:", len(chunks))
print("context length (chars):", len(context))


number of chunks used: 17
context length (chars): 16270


## Opportunity agent

Asks the local model to read the context and return a list of opportunities
as JSON, matching the fields the dashboard needs:
`title`, `impact_level`, `evidence`, `confidence_score`.


In [7]:
def opportunity_agent(context):
    prompt = f"""You are a business strategy analyst for SAP.

Read the news context below and identify 3 to 5 concrete business opportunities.
Opportunities can be about new markets, partnerships, AI products, or revenue growth.

Return ONLY a JSON array, no extra text, no markdown. Each item must look like this:

{{
  "title": "short opportunity title",
  "impact_level": "High, Medium, or Low",
  "evidence": "one sentence quoting or summarising the supporting news",
  "confidence_score": a number from 0 to 100
}}

Context:
{context}
"""
    raw = ask_llm(prompt)
    return raw


raw_output = opportunity_agent(context)
print(raw_output)


[
    {
      "title": "Leverage AI-Driven Discovery in Commerce",
      "impact_level": "High",
      "evidence": "Recently announced partnerships with companies such as Amazon Web Services, Google Cloud, Parloa, and Vercel enable new interaction models like conversational commerce.",
      "confidence_score": 85
    },
    {
      "title": "Expand Revenue Streams Through New Economic Models for Partners",
      "impact_level": "High",
      "evidence": "The economics for partners are changing. With Autonomous CX, partners can build: Industry solutions that can be reused and scaled Implementation packages that shorten delivery timelines Extensions and integrations across customers Ongoing services for AI optimization.",
      "confidence_score": 90
    },
    {
      "title": "Accelerate Digital Transformation with SAP's Composable Commerce Services",
      "impact_level": "Medium",
      "evidence": "SAP continues to be recognized in analyst evaluations, including the Gartner® Magic 

## Clean up the JSON

Small local models sometimes wrap the JSON in ```json fences or add a stray
sentence before/after it. This strips that out before parsing.


In [8]:
def parse_json_list(raw_text):
    text = raw_text.strip()
    # remove markdown code fences if the model added them
    if text.startswith("```"):
        text = text.strip("`")
        text = text.replace("json", "", 1).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print("could not parse JSON, here is the raw text:")
        print(text)
        return []


opportunities = parse_json_list(raw_output)
print("parsed", len(opportunities), "opportunities")
for opp in opportunities:
    print("-", opp.get("title"), "|", opp.get("impact_level"), "|", opp.get("confidence_score"))


parsed 3 opportunities
- Leverage AI-Driven Discovery in Commerce | High | 85
- Expand Revenue Streams Through New Economic Models for Partners | High | 90
- Accelerate Digital Transformation with SAP's Composable Commerce Services | Medium | 75


## Save results for the dashboard

In [9]:
with open(OUT_PATH, "w") as f:
    json.dump(opportunities, f, indent=2)

print("saved to", OUT_PATH)


saved to /workspaces/AI-CEO-Strategic-Intelligence-Agent/notebook/data/opportunities.json
